# 04 Methodology, Results, and Diagnostics

Refactored notebook: each model shows its Pooled OLS, Fixed Effects, Fixed Effects (Driscoll–Kraay), and Random Effects summaries directly.

## Table of Contents
1. [Econometric Methodology](#1-econometric-methodology)
2. [Setup](#2-setup)
3. [Model Results](#3-model-results) — one cell per model with all four estimator summaries
4. [Diagnostics](#4-diagnostics)
5. [Export](#5-export)


## 1. Econometric Methodology

### 1.1 Identification goal

Estimate how monetary-policy proxies are associated with FDI inflows across ASEAN country-year panels after controlling for country and year fixed effects.

$$
FDI_{it} = \alpha_i + \gamma_t + \beta X_{it-1} + \theta Z_{it} + \epsilon_{it}
$$

### 1.2 Estimators

| Estimator | When to use |
|---|---|
| Pooled OLS | Baseline, ignores panel structure |
| Fixed Effects (clustered) | Controls for unobserved heterogeneity |
| Fixed Effects (Driscoll–Kraay) | Adds cross-sectional dependence robustness |
| Random Effects | GLS; Hausman test decides vs FE |


## 2. Setup


In [136]:
import os
import importlib
from pathlib import Path
import sys
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

os.environ.setdefault('MPLCONFIGDIR', str(Path.cwd() / '.matplotlib_cache'))

try:
    import statsmodels.formula.api as smf
    from linearmodels.panel import PanelOLS, RandomEffects
except ModuleNotFoundError as exc:
    raise ModuleNotFoundError(
        'Install pandas, statsmodels, linearmodels, and scipy before running this notebook.'
    ) from exc

ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.panel_diagnostics import (
    hausman_test,
    pesaran_cd_test,
    pooled_model_diagnostics,
    vif_table,
)
from src.estimate_and_export import run_full_estimation_and_export
import src.model_contract as workbook_model_contract_module
workbook_model_contract_module = importlib.reload(workbook_model_contract_module)
from src.model_contract import (
    MODEL_ORDER,
    WORKBOOK_VARIABLE_MAP,
    add_country_lags,
    build_model_frame,
    build_workbook_model_catalog,
    choose_preferred_estimator,
    create_derived_columns,
    estimator_label,
    parse_regressor_string,
    safe_sheet_name,
)
from src.reporting import (
    VARIABLE_LABELS,
    THEORY_EXPECTED_SIGNS,
    stars,
    format_coef_cell,
    hausman_status,
    normalize_expected_sign,
    strip_lag_suffix,
    coefficient_direction,
    significance_label,
    sign_alignment,
    interpret_coefficient_row,
)
from src.config import FIGURES_DIR, OUTPUTS_DIR, PROCESSED_PANEL_FILE, ensure_output_dirs
from src.plot_helpers import correlation_heatmap, save_figure, setup_matplotlib_style

ensure_output_dirs()
setup_matplotlib_style()

PROCESSED_FILE = PROCESSED_PANEL_FILE
WORKBOOK_FILE = ROOT / 'model_selection_asean_fdi.xlsx'
DEPENDENT = 'fdi_pct_gdp'

pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', lambda value: f'{value:,.4f}')


In [137]:
# Load data and model catalog
df = pd.read_csv(PROCESSED_FILE)
df = create_derived_columns(df)
lag_cols = [col for col in df.columns if col not in ['country', 'year']]
df = add_country_lags(df, lag_cols, lag=1)

workbook_catalog_df, workbook_tables = build_workbook_model_catalog(
    WORKBOOK_FILE,
    panel_columns=df.columns.tolist(),
)
workbook_variable_selection = workbook_tables['variable_selection']
workbook_model_specs = workbook_tables['model_specs']
workbook_notes = workbook_tables['notes']

estimated_models = workbook_catalog_df[workbook_catalog_df['status'].eq('estimated')].copy()
estimated_models = estimated_models.sort_values('model_order').reset_index(drop=True)


## 3. Model Results

Each model below shows four estimator summaries: Pooled OLS, Fixed Effects (entity-clustered), Fixed Effects (Driscoll–Kraay), and Random Effects.


In [138]:
# M1_baseline_liquidity
model_id = 'M1_baseline_liquidity'
model_info = estimated_models.set_index('model_id').loc[model_id]
regressors = parse_regressor_string(model_info['mapped_regressors'])

estimation_frame = df.copy()
if model_info['lagged_model']:
    base_regressors = parse_regressor_string(model_info['base_regressors'])
    estimation_frame = add_country_lags(estimation_frame, base_regressors + [DEPENDENT], lag=1)

model_df = build_model_frame(
    estimation_frame,
    DEPENDENT,
    regressors,
    exclude_countries=[c.strip() for c in model_info['exclude_countries'].split(',')] if isinstance(model_info.get('exclude_countries'), str) and model_info['exclude_countries'].strip() else None
)
formula = DEPENDENT + ' ~ ' + ' + '.join(regressors)
panel_formula = DEPENDENT + ' ~ 1 + ' + ' + '.join(regressors)
reset_model_df = model_df.reset_index()

print(f'=== {model_id} ===')
print(f'Purpose: {model_info["purpose"]}')
print(f'Regressors: {", ".join(regressors)}')
print(f'Observations: {len(model_df)}')
print()

# 1. Pooled OLS
print('=' * 60)
print('Pooled OLS (country-clustered SE)')
print('=' * 60)
pooled = smf.ols(formula, data=reset_model_df).fit(
    cov_type='cluster', cov_kwds={'groups': reset_model_df['country']},
)
print(pooled.summary())

# 2. Hausman Decision & Conditional Estimation
print('=' * 60)
print('Hausman Test (FE vs RE)')
print('=' * 60)

re_model = None
try:
    re_model = RandomEffects.from_formula(panel_formula, data=model_df).fit(
        cov_type='clustered', cluster_entity=True,
    )
except (ZeroDivisionError, ValueError) as exc:
    pass

fe_clustered = PanelOLS.from_formula(
    panel_formula + ' + EntityEffects + TimeEffects', data=model_df,
).fit(cov_type='clustered', cluster_entity=True)

use_fe = True
if re_model is not None:
    try:
        h = hausman_test(fe_clustered, re_model, regressors)
        p_val = h['p_value']
        print(f"Hausman test statistic: {h['statistic']:.4f}")
        print(f"Hausman test p-value: {p_val:.4f}")
        if p_val >= 0.05:
            use_fe = False
            print("Recommendation: Fail to reject null hypothesis. Use Random Effects.")
        else:
            print("Recommendation: Reject null hypothesis. Use Fixed Effects.")
    except Exception as exc:
        print(f"Hausman test failed ({exc}). Defaulting to Fixed Effects.")
else:
    print("Random Effects estimation failed. Defaulting to Fixed Effects.")

if use_fe:
    print('=' * 60)
    print('Fixed Effects (entity-clustered SE)')
    print('=' * 60)
    print(fe_clustered.summary)

    print('=' * 60)
    print('Fixed Effects (Driscoll-Kraay SE)')
    print('=' * 60)
    fe_dk = PanelOLS.from_formula(
        panel_formula + ' + EntityEffects + TimeEffects', data=model_df,
    ).fit(cov_type='kernel', kernel='bartlett')
    print(fe_dk.summary)
else:
    print('=' * 60)
    print('Random Effects (entity-clustered SE)')
    print('=' * 60)
    print(re_model.summary)


=== M1_baseline_liquidity ===
Purpose: Kiểm tra kênh cung tiền
Regressors: broad_money_pct_gdp, inflation_gdp_deflator_pct, trade_pct_gdp, ln_gdppc, xr_dep_pct
Observations: 112

Pooled OLS (country-clustered SE)
                            OLS Regression Results                            
Dep. Variable:            fdi_pct_gdp   R-squared:                       0.695
Model:                            OLS   Adj. R-squared:                  0.681
Method:                 Least Squares   F-statistic:                     42.48
Date:                Thu, 21 May 2026   Prob (F-statistic):           1.50e-05
Time:                        23:52:41   Log-Likelihood:                -303.95
No. Observations:                 112   AIC:                             619.9
Df Residuals:                     106   BIC:                             636.2
Df Model:                           5                                         
Covariance Type:              cluster                                       

In [139]:
# M2_main_monetary_policy
model_id = 'M2_main_monetary_policy'
model_info = estimated_models.set_index('model_id').loc[model_id]
regressors = parse_regressor_string(model_info['mapped_regressors'])

estimation_frame = df.copy()
if model_info['lagged_model']:
    base_regressors = parse_regressor_string(model_info['base_regressors'])
    estimation_frame = add_country_lags(estimation_frame, base_regressors + [DEPENDENT], lag=1)

model_df = build_model_frame(
    estimation_frame,
    DEPENDENT,
    regressors,
    exclude_countries=[c.strip() for c in model_info['exclude_countries'].split(',')] if isinstance(model_info.get('exclude_countries'), str) and model_info['exclude_countries'].strip() else None
)
formula = DEPENDENT + ' ~ ' + ' + '.join(regressors)
panel_formula = DEPENDENT + ' ~ 1 + ' + ' + '.join(regressors)
reset_model_df = model_df.reset_index()

print(f'=== {model_id} ===')
print(f'Purpose: {model_info["purpose"]}')
print(f'Regressors: {", ".join(regressors)}')
print(f'Observations: {len(model_df)}')
print()

# 1. Pooled OLS
print('=' * 60)
print('Pooled OLS (country-clustered SE)')
print('=' * 60)
pooled = smf.ols(formula, data=reset_model_df).fit(
    cov_type='cluster', cov_kwds={'groups': reset_model_df['country']},
)
print(pooled.summary())

# 2. Hausman Decision & Conditional Estimation
print('=' * 60)
print('Hausman Test (FE vs RE)')
print('=' * 60)

re_model = None
try:
    re_model = RandomEffects.from_formula(panel_formula, data=model_df).fit(
        cov_type='clustered', cluster_entity=True,
    )
except (ZeroDivisionError, ValueError) as exc:
    pass

fe_clustered = PanelOLS.from_formula(
    panel_formula + ' + EntityEffects + TimeEffects', data=model_df,
).fit(cov_type='clustered', cluster_entity=True)

use_fe = True
if re_model is not None:
    try:
        h = hausman_test(fe_clustered, re_model, regressors)
        p_val = h['p_value']
        print(f"Hausman test statistic: {h['statistic']:.4f}")
        print(f"Hausman test p-value: {p_val:.4f}")
        if p_val >= 0.05:
            use_fe = False
            print("Recommendation: Fail to reject null hypothesis. Use Random Effects.")
        else:
            print("Recommendation: Reject null hypothesis. Use Fixed Effects.")
    except Exception as exc:
        print(f"Hausman test failed ({exc}). Defaulting to Fixed Effects.")
else:
    print("Random Effects estimation failed. Defaulting to Fixed Effects.")

if use_fe:
    print('=' * 60)
    print('Fixed Effects (entity-clustered SE)')
    print('=' * 60)
    print(fe_clustered.summary)

    print('=' * 60)
    print('Fixed Effects (Driscoll-Kraay SE)')
    print('=' * 60)
    fe_dk = PanelOLS.from_formula(
        panel_formula + ' + EntityEffects + TimeEffects', data=model_df,
    ).fit(cov_type='kernel', kernel='bartlett')
    print(fe_dk.summary)
else:
    print('=' * 60)
    print('Random Effects (entity-clustered SE)')
    print('=' * 60)
    print(re_model.summary)


=== M2_main_monetary_policy ===
Purpose: Kênh cung tiền + lãi suất
Regressors: broad_money_pct_gdp, deposit_interest_rate_pct, inflation_gdp_deflator_pct, trade_pct_gdp, ln_gdppc, xr_dep_pct
Observations: 96

Pooled OLS (country-clustered SE)
                            OLS Regression Results                            
Dep. Variable:            fdi_pct_gdp   R-squared:                       0.756
Model:                            OLS   Adj. R-squared:                  0.739
Method:                 Least Squares   F-statistic:                     320.1
Date:                Thu, 21 May 2026   Prob (F-statistic):           3.55e-08
Time:                        23:52:41   Log-Likelihood:                -253.95
No. Observations:                  96   AIC:                             521.9
Df Residuals:                      89   BIC:                             539.9
Df Model:                           6                                         
Covariance Type:              cluster         

In [140]:
# M3_lagged_main_model
model_id = 'M3_lagged_main_model'
model_info = estimated_models.set_index('model_id').loc[model_id]
regressors = parse_regressor_string(model_info['mapped_regressors'])

estimation_frame = df.copy()
if model_info['lagged_model']:
    base_regressors = parse_regressor_string(model_info['base_regressors'])
    estimation_frame = add_country_lags(estimation_frame, base_regressors + [DEPENDENT], lag=1)

model_df = build_model_frame(
    estimation_frame,
    DEPENDENT,
    regressors,
    exclude_countries=[c.strip() for c in model_info['exclude_countries'].split(',')] if isinstance(model_info.get('exclude_countries'), str) and model_info['exclude_countries'].strip() else None
)
formula = DEPENDENT + ' ~ ' + ' + '.join(regressors)
panel_formula = DEPENDENT + ' ~ 1 + ' + ' + '.join(regressors)
reset_model_df = model_df.reset_index()

print(f'=== {model_id} ===')
print(f'Purpose: {model_info["purpose"]}')
print(f'Regressors: {", ".join(regressors)}')
print(f'Observations: {len(model_df)}')
print()

# 1. Pooled OLS
print('=' * 60)
print('Pooled OLS (country-clustered SE)')
print('=' * 60)
pooled = smf.ols(formula, data=reset_model_df).fit(
    cov_type='cluster', cov_kwds={'groups': reset_model_df['country']},
)
print(pooled.summary())

# 2. Hausman Decision & Conditional Estimation
print('=' * 60)
print('Hausman Test (FE vs RE)')
print('=' * 60)

re_model = None
try:
    re_model = RandomEffects.from_formula(panel_formula, data=model_df).fit(
        cov_type='clustered', cluster_entity=True,
    )
except (ZeroDivisionError, ValueError) as exc:
    pass

fe_clustered = PanelOLS.from_formula(
    panel_formula + ' + EntityEffects + TimeEffects', data=model_df,
).fit(cov_type='clustered', cluster_entity=True)

use_fe = True
if re_model is not None:
    try:
        h = hausman_test(fe_clustered, re_model, regressors)
        p_val = h['p_value']
        print(f"Hausman test statistic: {h['statistic']:.4f}")
        print(f"Hausman test p-value: {p_val:.4f}")
        if p_val >= 0.05:
            use_fe = False
            print("Recommendation: Fail to reject null hypothesis. Use Random Effects.")
        else:
            print("Recommendation: Reject null hypothesis. Use Fixed Effects.")
    except Exception as exc:
        print(f"Hausman test failed ({exc}). Defaulting to Fixed Effects.")
else:
    print("Random Effects estimation failed. Defaulting to Fixed Effects.")

if use_fe:
    print('=' * 60)
    print('Fixed Effects (entity-clustered SE)')
    print('=' * 60)
    print(fe_clustered.summary)

    print('=' * 60)
    print('Fixed Effects (Driscoll-Kraay SE)')
    print('=' * 60)
    fe_dk = PanelOLS.from_formula(
        panel_formula + ' + EntityEffects + TimeEffects', data=model_df,
    ).fit(cov_type='kernel', kernel='bartlett')
    print(fe_dk.summary)
else:
    print('=' * 60)
    print('Random Effects (entity-clustered SE)')
    print('=' * 60)
    print(re_model.summary)


=== M3_lagged_main_model ===
Purpose: Giảm vấn đề nội sinh/đồng thời
Regressors: broad_money_pct_gdp_lag1, deposit_interest_rate_pct_lag1, inflation_gdp_deflator_pct_lag1, trade_pct_gdp_lag1, ln_gdppc_lag1, xr_dep_pct_lag1
Observations: 91

Pooled OLS (country-clustered SE)
                            OLS Regression Results                            
Dep. Variable:            fdi_pct_gdp   R-squared:                       0.766
Model:                            OLS   Adj. R-squared:                  0.749
Method:                 Least Squares   F-statistic:                     55.52
Date:                Thu, 21 May 2026   Prob (F-statistic):           1.50e-05
Time:                        23:52:41   Log-Likelihood:                -246.79
No. Observations:                  91   AIC:                             507.6
Df Residuals:                      84   BIC:                             525.2
Df Model:                           6                                         
Covariance Typ

In [141]:
# M4_real_interest_robustness
model_id = 'M4_real_interest_robustness'
model_info = estimated_models.set_index('model_id').loc[model_id]
regressors = parse_regressor_string(model_info['mapped_regressors'])

estimation_frame = df.copy()
if model_info['lagged_model']:
    base_regressors = parse_regressor_string(model_info['base_regressors'])
    estimation_frame = add_country_lags(estimation_frame, base_regressors + [DEPENDENT], lag=1)

model_df = build_model_frame(
    estimation_frame,
    DEPENDENT,
    regressors,
    exclude_countries=[c.strip() for c in model_info['exclude_countries'].split(',')] if isinstance(model_info.get('exclude_countries'), str) and model_info['exclude_countries'].strip() else None
)
formula = DEPENDENT + ' ~ ' + ' + '.join(regressors)
panel_formula = DEPENDENT + ' ~ 1 + ' + ' + '.join(regressors)
reset_model_df = model_df.reset_index()

print(f'=== {model_id} ===')
print(f'Purpose: {model_info["purpose"]}')
print(f'Regressors: {", ".join(regressors)}')
print(f'Observations: {len(model_df)}')
print()

# 1. Pooled OLS
print('=' * 60)
print('Pooled OLS (country-clustered SE)')
print('=' * 60)
pooled = smf.ols(formula, data=reset_model_df).fit(
    cov_type='cluster', cov_kwds={'groups': reset_model_df['country']},
)
print(pooled.summary())

# 2. Hausman Decision & Conditional Estimation
print('=' * 60)
print('Hausman Test (FE vs RE)')
print('=' * 60)

re_model = None
try:
    re_model = RandomEffects.from_formula(panel_formula, data=model_df).fit(
        cov_type='clustered', cluster_entity=True,
    )
except (ZeroDivisionError, ValueError) as exc:
    pass

fe_clustered = PanelOLS.from_formula(
    panel_formula + ' + EntityEffects + TimeEffects', data=model_df,
).fit(cov_type='clustered', cluster_entity=True)

use_fe = True
if re_model is not None:
    try:
        h = hausman_test(fe_clustered, re_model, regressors)
        p_val = h['p_value']
        print(f"Hausman test statistic: {h['statistic']:.4f}")
        print(f"Hausman test p-value: {p_val:.4f}")
        if p_val >= 0.05:
            use_fe = False
            print("Recommendation: Fail to reject null hypothesis. Use Random Effects.")
        else:
            print("Recommendation: Reject null hypothesis. Use Fixed Effects.")
    except Exception as exc:
        print(f"Hausman test failed ({exc}). Defaulting to Fixed Effects.")
else:
    print("Random Effects estimation failed. Defaulting to Fixed Effects.")

if use_fe:
    print('=' * 60)
    print('Fixed Effects (entity-clustered SE)')
    print('=' * 60)
    print(fe_clustered.summary)

    print('=' * 60)
    print('Fixed Effects (Driscoll-Kraay SE)')
    print('=' * 60)
    fe_dk = PanelOLS.from_formula(
        panel_formula + ' + EntityEffects + TimeEffects', data=model_df,
    ).fit(cov_type='kernel', kernel='bartlett')
    print(fe_dk.summary)
else:
    print('=' * 60)
    print('Random Effects (entity-clustered SE)')
    print('=' * 60)
    print(re_model.summary)


=== M4_real_interest_robustness ===
Purpose: Thay proxy lãi suất bằng real interest rate
Regressors: broad_money_pct_gdp, real_interest_rate_pct, trade_pct_gdp, ln_gdppc, xr_dep_pct
Observations: 96

Pooled OLS (country-clustered SE)
                            OLS Regression Results                            
Dep. Variable:            fdi_pct_gdp   R-squared:                       0.739
Model:                            OLS   Adj. R-squared:                  0.724
Method:                 Least Squares   F-statistic:                     32.51
Date:                Thu, 21 May 2026   Prob (F-statistic):           0.000106
Time:                        23:52:41   Log-Likelihood:                -257.13
No. Observations:                  96   AIC:                             526.3
Df Residuals:                      90   BIC:                             541.7
Df Model:                           5                                         
Covariance Type:              cluster                  

In [142]:
# M5_lending_rate_robustness
model_id = 'M5_lending_rate_robustness'
model_info = estimated_models.set_index('model_id').loc[model_id]
regressors = parse_regressor_string(model_info['mapped_regressors'])

estimation_frame = df.copy()
if model_info['lagged_model']:
    base_regressors = parse_regressor_string(model_info['base_regressors'])
    estimation_frame = add_country_lags(estimation_frame, base_regressors + [DEPENDENT], lag=1)

model_df = build_model_frame(
    estimation_frame,
    DEPENDENT,
    regressors,
    exclude_countries=[c.strip() for c in model_info['exclude_countries'].split(',')] if isinstance(model_info.get('exclude_countries'), str) and model_info['exclude_countries'].strip() else None
)
formula = DEPENDENT + ' ~ ' + ' + '.join(regressors)
panel_formula = DEPENDENT + ' ~ 1 + ' + ' + '.join(regressors)
reset_model_df = model_df.reset_index()

print(f'=== {model_id} ===')
print(f'Purpose: {model_info["purpose"]}')
print(f'Regressors: {", ".join(regressors)}')
print(f'Observations: {len(model_df)}')
print()

# 1. Pooled OLS
print('=' * 60)
print('Pooled OLS (country-clustered SE)')
print('=' * 60)
pooled = smf.ols(formula, data=reset_model_df).fit(
    cov_type='cluster', cov_kwds={'groups': reset_model_df['country']},
)
print(pooled.summary())

# 2. Hausman Decision & Conditional Estimation
print('=' * 60)
print('Hausman Test (FE vs RE)')
print('=' * 60)

re_model = None
try:
    re_model = RandomEffects.from_formula(panel_formula, data=model_df).fit(
        cov_type='clustered', cluster_entity=True,
    )
except (ZeroDivisionError, ValueError) as exc:
    pass

fe_clustered = PanelOLS.from_formula(
    panel_formula + ' + EntityEffects + TimeEffects', data=model_df,
).fit(cov_type='clustered', cluster_entity=True)

use_fe = True
if re_model is not None:
    try:
        h = hausman_test(fe_clustered, re_model, regressors)
        p_val = h['p_value']
        print(f"Hausman test statistic: {h['statistic']:.4f}")
        print(f"Hausman test p-value: {p_val:.4f}")
        if p_val >= 0.05:
            use_fe = False
            print("Recommendation: Fail to reject null hypothesis. Use Random Effects.")
        else:
            print("Recommendation: Reject null hypothesis. Use Fixed Effects.")
    except Exception as exc:
        print(f"Hausman test failed ({exc}). Defaulting to Fixed Effects.")
else:
    print("Random Effects estimation failed. Defaulting to Fixed Effects.")

if use_fe:
    print('=' * 60)
    print('Fixed Effects (entity-clustered SE)')
    print('=' * 60)
    print(fe_clustered.summary)

    print('=' * 60)
    print('Fixed Effects (Driscoll-Kraay SE)')
    print('=' * 60)
    fe_dk = PanelOLS.from_formula(
        panel_formula + ' + EntityEffects + TimeEffects', data=model_df,
    ).fit(cov_type='kernel', kernel='bartlett')
    print(fe_dk.summary)
else:
    print('=' * 60)
    print('Random Effects (entity-clustered SE)')
    print('=' * 60)
    print(re_model.summary)


=== M5_lending_rate_robustness ===
Purpose: Thay proxy lãi suất bằng lending rate
Regressors: broad_money_pct_gdp, lending_interest_rate_pct, inflation_gdp_deflator_pct, trade_pct_gdp, ln_gdppc, xr_dep_pct
Observations: 96

Pooled OLS (country-clustered SE)
                            OLS Regression Results                            
Dep. Variable:            fdi_pct_gdp   R-squared:                       0.747
Model:                            OLS   Adj. R-squared:                  0.729
Method:                 Least Squares   F-statistic:                     37.19
Date:                Thu, 21 May 2026   Prob (F-statistic):           5.79e-05
Time:                        23:52:41   Log-Likelihood:                -255.70
No. Observations:                  96   AIC:                             525.4
Df Residuals:                      89   BIC:                             543.3
Df Model:                           6                                         
Covariance Type:              c

In [143]:
# M6a_tourism_robustness_from_M2
model_id = 'M6a_tourism_robustness_from_M2'
model_info = estimated_models.set_index('model_id').loc[model_id]
regressors = parse_regressor_string(model_info['mapped_regressors'])

estimation_frame = df.copy()
if model_info['lagged_model']:
    base_regressors = parse_regressor_string(model_info['base_regressors'])
    estimation_frame = add_country_lags(estimation_frame, base_regressors + [DEPENDENT], lag=1)

model_df = build_model_frame(
    estimation_frame,
    DEPENDENT,
    regressors,
    exclude_countries=[c.strip() for c in model_info['exclude_countries'].split(',')] if isinstance(model_info.get('exclude_countries'), str) and model_info['exclude_countries'].strip() else None
)
formula = DEPENDENT + ' ~ ' + ' + '.join(regressors)
panel_formula = DEPENDENT + ' ~ 1 + ' + ' + '.join(regressors)
reset_model_df = model_df.reset_index()

print(f'=== {model_id} ===')
print(f'Purpose: {model_info["purpose"]}')
print(f'Regressors: {", ".join(regressors)}')
print(f'Observations: {len(model_df)}')
print()

# 1. Pooled OLS
print('=' * 60)
print('Pooled OLS (country-clustered SE)')
print('=' * 60)
pooled = smf.ols(formula, data=reset_model_df).fit(
    cov_type='cluster', cov_kwds={'groups': reset_model_df['country']},
)
print(pooled.summary())

# 2. Hausman Decision & Conditional Estimation
print('=' * 60)
print('Hausman Test (FE vs RE)')
print('=' * 60)

re_model = None
try:
    re_model = RandomEffects.from_formula(panel_formula, data=model_df).fit(
        cov_type='clustered', cluster_entity=True,
    )
except (ZeroDivisionError, ValueError) as exc:
    pass

fe_clustered = PanelOLS.from_formula(
    panel_formula + ' + EntityEffects + TimeEffects', data=model_df,
).fit(cov_type='clustered', cluster_entity=True)

use_fe = True
if re_model is not None:
    try:
        h = hausman_test(fe_clustered, re_model, regressors)
        p_val = h['p_value']
        print(f"Hausman test statistic: {h['statistic']:.4f}")
        print(f"Hausman test p-value: {p_val:.4f}")
        if p_val >= 0.05:
            use_fe = False
            print("Recommendation: Fail to reject null hypothesis. Use Random Effects.")
        else:
            print("Recommendation: Reject null hypothesis. Use Fixed Effects.")
    except Exception as exc:
        print(f"Hausman test failed ({exc}). Defaulting to Fixed Effects.")
else:
    print("Random Effects estimation failed. Defaulting to Fixed Effects.")

if use_fe:
    print('=' * 60)
    print('Fixed Effects (entity-clustered SE)')
    print('=' * 60)
    print(fe_clustered.summary)

    print('=' * 60)
    print('Fixed Effects (Driscoll-Kraay SE)')
    print('=' * 60)
    fe_dk = PanelOLS.from_formula(
        panel_formula + ' + EntityEffects + TimeEffects', data=model_df,
    ).fit(cov_type='kernel', kernel='bartlett')
    print(fe_dk.summary)
else:
    print('=' * 60)
    print('Random Effects (entity-clustered SE)')
    print('=' * 60)
    print(re_model.summary)


=== M6a_tourism_robustness_from_M2 ===
Purpose: Kiểm tra kênh dịch vụ/du lịch
Regressors: broad_money_pct_gdp, deposit_interest_rate_pct, inflation_gdp_deflator_pct, trade_pct_gdp, ln_gdppc, xr_dep_pct, ln_tourism_arrivals
Observations: 96

Pooled OLS (country-clustered SE)
                            OLS Regression Results                            
Dep. Variable:            fdi_pct_gdp   R-squared:                       0.778
Model:                            OLS   Adj. R-squared:                  0.760
Method:                 Least Squares   F-statistic:                     2043.
Date:                Thu, 21 May 2026   Prob (F-statistic):           4.82e-11
Time:                        23:52:42   Log-Likelihood:                -249.42
No. Observations:                  96   AIC:                             514.8
Df Residuals:                      88   BIC:                             535.4
Df Model:                           7                                         
Covariance Typ

In [144]:
# M6b_tourism_robustness_from_M4
model_id = 'M6b_tourism_robustness_from_M4'
model_info = estimated_models.set_index('model_id').loc[model_id]
regressors = parse_regressor_string(model_info['mapped_regressors'])

estimation_frame = df.copy()
if model_info['lagged_model']:
    base_regressors = parse_regressor_string(model_info['base_regressors'])
    estimation_frame = add_country_lags(estimation_frame, base_regressors + [DEPENDENT], lag=1)

model_df = build_model_frame(
    estimation_frame,
    DEPENDENT,
    regressors,
    exclude_countries=[c.strip() for c in model_info['exclude_countries'].split(',')] if isinstance(model_info.get('exclude_countries'), str) and model_info['exclude_countries'].strip() else None
)
formula = DEPENDENT + ' ~ ' + ' + '.join(regressors)
panel_formula = DEPENDENT + ' ~ 1 + ' + ' + '.join(regressors)
reset_model_df = model_df.reset_index()

print(f'=== {model_id} ===')
print(f'Purpose: {model_info["purpose"]}')
print(f'Regressors: {", ".join(regressors)}')
print(f'Observations: {len(model_df)}')
print()

# 1. Pooled OLS
print('=' * 60)
print('Pooled OLS (country-clustered SE)')
print('=' * 60)
pooled = smf.ols(formula, data=reset_model_df).fit(
    cov_type='cluster', cov_kwds={'groups': reset_model_df['country']},
)
print(pooled.summary())

# 2. Hausman Decision & Conditional Estimation
print('=' * 60)
print('Hausman Test (FE vs RE)')
print('=' * 60)

re_model = None
try:
    re_model = RandomEffects.from_formula(panel_formula, data=model_df).fit(
        cov_type='clustered', cluster_entity=True,
    )
except (ZeroDivisionError, ValueError) as exc:
    pass

fe_clustered = PanelOLS.from_formula(
    panel_formula + ' + EntityEffects + TimeEffects', data=model_df,
).fit(cov_type='clustered', cluster_entity=True)

use_fe = True
if re_model is not None:
    try:
        h = hausman_test(fe_clustered, re_model, regressors)
        p_val = h['p_value']
        print(f"Hausman test statistic: {h['statistic']:.4f}")
        print(f"Hausman test p-value: {p_val:.4f}")
        if p_val >= 0.05:
            use_fe = False
            print("Recommendation: Fail to reject null hypothesis. Use Random Effects.")
        else:
            print("Recommendation: Reject null hypothesis. Use Fixed Effects.")
    except Exception as exc:
        print(f"Hausman test failed ({exc}). Defaulting to Fixed Effects.")
else:
    print("Random Effects estimation failed. Defaulting to Fixed Effects.")

if use_fe:
    print('=' * 60)
    print('Fixed Effects (entity-clustered SE)')
    print('=' * 60)
    print(fe_clustered.summary)

    print('=' * 60)
    print('Fixed Effects (Driscoll-Kraay SE)')
    print('=' * 60)
    fe_dk = PanelOLS.from_formula(
        panel_formula + ' + EntityEffects + TimeEffects', data=model_df,
    ).fit(cov_type='kernel', kernel='bartlett')
    print(fe_dk.summary)
else:
    print('=' * 60)
    print('Random Effects (entity-clustered SE)')
    print('=' * 60)
    print(re_model.summary)


=== M6b_tourism_robustness_from_M4 ===
Purpose: Kiểm tra kênh dịch vụ/du lịch
Regressors: broad_money_pct_gdp, real_interest_rate_pct, trade_pct_gdp, ln_gdppc, xr_dep_pct, ln_tourism_arrivals
Observations: 96

Pooled OLS (country-clustered SE)


                            OLS Regression Results                            
Dep. Variable:            fdi_pct_gdp   R-squared:                       0.772
Model:                            OLS   Adj. R-squared:                  0.757
Method:                 Least Squares   F-statistic:                     284.7
Date:                Thu, 21 May 2026   Prob (F-statistic):           5.34e-08
Time:                        23:52:42   Log-Likelihood:                -250.62
No. Observations:                  96   AIC:                             515.2
Df Residuals:                      89   BIC:                             533.2
Df Model:                           6                                         
Covariance Type:              cluster                                         
                             coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------------------
Intercept                -16

In [145]:
# M7a_human_capital_robustness_from_M2
model_id = 'M7a_human_capital_robustness_from_M2'
model_info = estimated_models.set_index('model_id').loc[model_id]
regressors = parse_regressor_string(model_info['mapped_regressors'])

estimation_frame = df.copy()
if model_info['lagged_model']:
    base_regressors = parse_regressor_string(model_info['base_regressors'])
    estimation_frame = add_country_lags(estimation_frame, base_regressors + [DEPENDENT], lag=1)

model_df = build_model_frame(
    estimation_frame,
    DEPENDENT,
    regressors,
    exclude_countries=[c.strip() for c in model_info['exclude_countries'].split(',')] if isinstance(model_info.get('exclude_countries'), str) and model_info['exclude_countries'].strip() else None
)
formula = DEPENDENT + ' ~ ' + ' + '.join(regressors)
panel_formula = DEPENDENT + ' ~ 1 + ' + ' + '.join(regressors)
reset_model_df = model_df.reset_index()

print(f'=== {model_id} ===')
print(f'Purpose: {model_info["purpose"]}')
print(f'Regressors: {", ".join(regressors)}')
print(f'Observations: {len(model_df)}')
print()

# 1. Pooled OLS
print('=' * 60)
print('Pooled OLS (country-clustered SE)')
print('=' * 60)
pooled = smf.ols(formula, data=reset_model_df).fit(
    cov_type='cluster', cov_kwds={'groups': reset_model_df['country']},
)
print(pooled.summary())

# 2. Hausman Decision & Conditional Estimation
print('=' * 60)
print('Hausman Test (FE vs RE)')
print('=' * 60)

re_model = None
try:
    re_model = RandomEffects.from_formula(panel_formula, data=model_df).fit(
        cov_type='clustered', cluster_entity=True,
    )
except (ZeroDivisionError, ValueError) as exc:
    pass

fe_clustered = PanelOLS.from_formula(
    panel_formula + ' + EntityEffects + TimeEffects', data=model_df,
).fit(cov_type='clustered', cluster_entity=True)

use_fe = True
if re_model is not None:
    try:
        h = hausman_test(fe_clustered, re_model, regressors)
        p_val = h['p_value']
        print(f"Hausman test statistic: {h['statistic']:.4f}")
        print(f"Hausman test p-value: {p_val:.4f}")
        if p_val >= 0.05:
            use_fe = False
            print("Recommendation: Fail to reject null hypothesis. Use Random Effects.")
        else:
            print("Recommendation: Reject null hypothesis. Use Fixed Effects.")
    except Exception as exc:
        print(f"Hausman test failed ({exc}). Defaulting to Fixed Effects.")
else:
    print("Random Effects estimation failed. Defaulting to Fixed Effects.")

if use_fe:
    print('=' * 60)
    print('Fixed Effects (entity-clustered SE)')
    print('=' * 60)
    print(fe_clustered.summary)

    print('=' * 60)
    print('Fixed Effects (Driscoll-Kraay SE)')
    print('=' * 60)
    fe_dk = PanelOLS.from_formula(
        panel_formula + ' + EntityEffects + TimeEffects', data=model_df,
    ).fit(cov_type='kernel', kernel='bartlett')
    print(fe_dk.summary)
else:
    print('=' * 60)
    print('Random Effects (entity-clustered SE)')
    print('=' * 60)
    print(re_model.summary)


=== M7a_human_capital_robustness_from_M2 ===
Purpose: Kiểm soát chất lượng lao động
Regressors: broad_money_pct_gdp, deposit_interest_rate_pct, inflation_gdp_deflator_pct, trade_pct_gdp, ln_gdppc, xr_dep_pct, hc_human_capital_index
Observations: 83

Pooled OLS (country-clustered SE)
                            OLS Regression Results                            
Dep. Variable:            fdi_pct_gdp   R-squared:                       0.861
Model:                            OLS   Adj. R-squared:                  0.848
Method:                 Least Squares   F-statistic:                     61.61
Date:                Thu, 21 May 2026   Prob (F-statistic):           3.98e-05
Time:                        23:52:42   Log-Likelihood:                -194.27
No. Observations:                  83   AIC:                             404.5
Df Residuals:                      75   BIC:                             423.9
Df Model:                           7                                         
Covar

/Users/bunnypro/miniconda3/lib/python3.13/site-packages/statsmodels/base/model.py:1894: ValueWarning: covariance of constraints does not have full rank. The number of constraints is 7, but rank is 6
  warnings.warn('covariance of constraints does not have full '


In [146]:
# M7b_human_capital_robustness_from_M4
model_id = 'M7b_human_capital_robustness_from_M4'
model_info = estimated_models.set_index('model_id').loc[model_id]
regressors = parse_regressor_string(model_info['mapped_regressors'])

estimation_frame = df.copy()
if model_info['lagged_model']:
    base_regressors = parse_regressor_string(model_info['base_regressors'])
    estimation_frame = add_country_lags(estimation_frame, base_regressors + [DEPENDENT], lag=1)

model_df = build_model_frame(
    estimation_frame,
    DEPENDENT,
    regressors,
    exclude_countries=[c.strip() for c in model_info['exclude_countries'].split(',')] if isinstance(model_info.get('exclude_countries'), str) and model_info['exclude_countries'].strip() else None
)
formula = DEPENDENT + ' ~ ' + ' + '.join(regressors)
panel_formula = DEPENDENT + ' ~ 1 + ' + ' + '.join(regressors)
reset_model_df = model_df.reset_index()

print(f'=== {model_id} ===')
print(f'Purpose: {model_info["purpose"]}')
print(f'Regressors: {", ".join(regressors)}')
print(f'Observations: {len(model_df)}')
print()

# 1. Pooled OLS
print('=' * 60)
print('Pooled OLS (country-clustered SE)')
print('=' * 60)
pooled = smf.ols(formula, data=reset_model_df).fit(
    cov_type='cluster', cov_kwds={'groups': reset_model_df['country']},
)
print(pooled.summary())

# 2. Hausman Decision & Conditional Estimation
print('=' * 60)
print('Hausman Test (FE vs RE)')
print('=' * 60)

re_model = None
try:
    re_model = RandomEffects.from_formula(panel_formula, data=model_df).fit(
        cov_type='clustered', cluster_entity=True,
    )
except (ZeroDivisionError, ValueError) as exc:
    pass

fe_clustered = PanelOLS.from_formula(
    panel_formula + ' + EntityEffects + TimeEffects', data=model_df,
).fit(cov_type='clustered', cluster_entity=True)

use_fe = True
if re_model is not None:
    try:
        h = hausman_test(fe_clustered, re_model, regressors)
        p_val = h['p_value']
        print(f"Hausman test statistic: {h['statistic']:.4f}")
        print(f"Hausman test p-value: {p_val:.4f}")
        if p_val >= 0.05:
            use_fe = False
            print("Recommendation: Fail to reject null hypothesis. Use Random Effects.")
        else:
            print("Recommendation: Reject null hypothesis. Use Fixed Effects.")
    except Exception as exc:
        print(f"Hausman test failed ({exc}). Defaulting to Fixed Effects.")
else:
    print("Random Effects estimation failed. Defaulting to Fixed Effects.")

if use_fe:
    print('=' * 60)
    print('Fixed Effects (entity-clustered SE)')
    print('=' * 60)
    print(fe_clustered.summary)

    print('=' * 60)
    print('Fixed Effects (Driscoll-Kraay SE)')
    print('=' * 60)
    fe_dk = PanelOLS.from_formula(
        panel_formula + ' + EntityEffects + TimeEffects', data=model_df,
    ).fit(cov_type='kernel', kernel='bartlett')
    print(fe_dk.summary)
else:
    print('=' * 60)
    print('Random Effects (entity-clustered SE)')
    print('=' * 60)
    print(re_model.summary)


=== M7b_human_capital_robustness_from_M4 ===
Purpose: Kiểm soát chất lượng lao động
Regressors: broad_money_pct_gdp, real_interest_rate_pct, trade_pct_gdp, ln_gdppc, xr_dep_pct, hc_human_capital_index
Observations: 83

Pooled OLS (country-clustered SE)
                            OLS Regression Results                            
Dep. Variable:            fdi_pct_gdp   R-squared:                       0.859
Model:                            OLS   Adj. R-squared:                  0.848
Method:                 Least Squares   F-statistic:                     673.9
Date:                Thu, 21 May 2026   Prob (F-statistic):           3.25e-08
Time:                        23:52:42   Log-Likelihood:                -194.90
No. Observations:                  83   AIC:                             403.8
Df Residuals:                      76   BIC:                             420.7
Df Model:                           6                                         
Covariance Type:              cluste

## 4. Diagnostics

Hausman test, VIF, and Pesaran CD for each model.


In [147]:
# Run diagnostics for all models and collect results
hausman_rows, vif_rows, diag_rows = [], [], []

for model_row in estimated_models.itertuples(index=False):
    model_id = model_row.model_id
    regressors = parse_regressor_string(model_row.mapped_regressors)
    estimation_frame = df.copy()
    if model_row.lagged_model:
        base_regressors = parse_regressor_string(model_row.base_regressors)
        estimation_frame = add_country_lags(estimation_frame, base_regressors + [DEPENDENT], lag=1)
    exclude_countries = [c.strip() for c in model_row.exclude_countries.split(',')] if isinstance(getattr(model_row, 'exclude_countries', None), str) and model_row.exclude_countries.strip() else None
    model_df = build_model_frame(estimation_frame, DEPENDENT, regressors, exclude_countries=exclude_countries)
    panel_formula = DEPENDENT + ' ~ 1 + ' + ' + '.join(regressors)
    reset_model_df = model_df.reset_index()

    fe_clustered = PanelOLS.from_formula(
        panel_formula + ' + EntityEffects + TimeEffects', data=model_df,
    ).fit(cov_type='clustered', cluster_entity=True)
    fe_dk = PanelOLS.from_formula(
        panel_formula + ' + EntityEffects + TimeEffects', data=model_df,
    ).fit(cov_type='kernel', kernel='bartlett')

    re = None
    try:
        re = RandomEffects.from_formula(panel_formula, data=model_df).fit(
            cov_type='clustered', cluster_entity=True,
        )
    except (ZeroDivisionError, ValueError):
        pass

    if re is not None:
        h = hausman_test(fe_clustered, re, regressors)
        hausman_rows.append({'model_id': model_id, **h})
    else:
        hausman_rows.append({
            'model_id': model_id, 'statistic': np.nan, 'p_value': np.nan,
            'degrees_of_freedom': len(regressors), 'negative_statistic_flag': False,
            'variables': ', '.join(regressors),
        })

    vif_df_model = vif_table(model_df, model_id, regressors)
    vif_rows.append(vif_df_model)

    formula = DEPENDENT + ' ~ ' + ' + '.join(regressors)
    pooled = smf.ols(formula, data=reset_model_df).fit(
        cov_type='cluster', cov_kwds={'groups': reset_model_df['country']},
    )
    d = pooled_model_diagnostics(pooled, model_df, model_id, regressors)
    d.update(pesaran_cd_test(fe_dk.resids, model_id))
    diag_rows.append(d)

hausman_df = pd.DataFrame(hausman_rows)
vif_all = pd.concat(vif_rows, ignore_index=True)
diagnostics_df = pd.DataFrame(diag_rows)

# Hausman summary
print('=' * 60)
print('Hausman Test Summary')
print('=' * 60)
hausman_summary = hausman_df[['model_id', 'statistic', 'p_value', 'degrees_of_freedom']].copy()
hausman_summary['recommendation'] = hausman_summary.apply(
    lambda r: 'FE' if r['p_value'] < 0.05 else 'RE' if not pd.isna(r['p_value']) else 'FE (RE failed)', axis=1,
)
print(hausman_summary.to_string(index=False))
print()

# VIF summary
print('=' * 60)
print('VIF Multicollinearity Summary')
print('=' * 60)
vif_summary = (
    vif_all.groupby('specification')['vif']
    .agg(['max', 'mean'])
    .rename(columns={'specification': 'model_id'})
    .reset_index()
)
print(vif_summary.to_string(index=False))
print()

# Pesaran CD
print('=' * 60)
print('Pesaran Cross-Sectional Dependence Test')
print('=' * 60)
print(diagnostics_df[['specification', 'pesaran_cd_stat', 'pesaran_cd_p_value']].to_string(index=False))



Hausman Test Summary
                            model_id  statistic  p_value  degrees_of_freedom recommendation
               M1_baseline_liquidity   265.9365   0.0000                   5             FE
             M2_main_monetary_policy     0.0000   1.0000                   6             RE
                M3_lagged_main_model     0.0000   1.0000                   6             RE
         M4_real_interest_robustness     0.0000   1.0000                   5             RE
          M5_lending_rate_robustness     5.5217   0.4788                   6             RE
      M6a_tourism_robustness_from_M2        NaN      NaN                   7 FE (RE failed)
      M6b_tourism_robustness_from_M4    74.2401   0.0000                   6             FE
M7a_human_capital_robustness_from_M2     0.0000   1.0000                   7             RE
M7b_human_capital_robustness_from_M4        NaN      NaN                   6 FE (RE failed)

VIF Multicollinearity Summary
                       speci

## 5. Export


In [148]:
# Run full estimation and export pipeline
run_full_estimation_and_export(df, estimated_models, WORKBOOK_FILE)
print('Regenerated all regression tables, diagnostic outputs, and Excel workbooks.')


--- Starting Full Econometric Estimation & Export Pipeline ---
Estimating native specifications...
Running model sample loss audits...
Running common sample estimations...
Running model panel balance summaries...
Writing main output CSV files...
Computing descriptive stats and correlations...
Building diagnostic tables and regression detail files for individual models...
Building main regression table...
Running stepwise broad money sign decomposition...
Running trade collinearity robustness drops...
Writing bulk workbook outputs/model_outputs.xlsx...
Writing bulk workbook outputs/results_tables.xlsx...
--- Econometric Estimation & Export Pipeline Completed Successfully ---
Regenerated all regression tables, diagnostic outputs, and Excel workbooks.
